In [ ]:
# --- KAGGLE "DISK DOSTU" GGUF DÖNÜŞTÜRME ---
# Bu script, 20GB limitine takılmamak için işlemi /tmp klasöründe yapar.

import os
import shutil

# 1. Gerekli Kütüphaneleri Kur
print("Kurulum yapılıyor...")
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes --quiet
!pip install gdown

Kurulum yapılıyor...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 28.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 36.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.1 MB/s eta 0:00:00:00:0100:0

In [ ]:
# 2. Çalışma Dizinini /tmp Olarak Değiştir (KRİTİK ADIM)
# /kaggle/working 20GB ile sınırlıdır, /tmp ise çok daha geniştir.
os.chdir("/tmp")
print("Çalışma dizini /tmp olarak değiştirildi.")

# 3. Adaptörü Drive'dan İndir
# BURAYA KENDİ DRIVE ID'Nİ YAZMAYI UNUTMA!
file_id = '1qDta8PKGAWzanCn8_HXcZY2YRwwxWVJp' 
url = f'https://drive.google.com/uc?id={file_id}'
output = 'finwise_scribe_adapter_v1.zip'

print(f"Dosya indiriliyor (ID: {file_id})...")
!gdown {url} -O {output}

# 4. Zip'i Çıkar
print("Zip açılıyor...")
!unzip -o {output} -d finwise_adapter

# 5. Modeli Yükle
from unsloth import FastLanguageModel
import torch

# Adaptör klasörünü bul
adapter_dir = "finwise_adapter"
for root, dirs, files in os.walk("finwise_adapter"):
    if "adapter_config.json" in files:
        adapter_dir = root
        break

print(f"Model yükleniyor: {adapter_dir}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_dir,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# 6. GGUF'a Dönüştür (/tmp içinde)
print("💾 GGUF formatına dönüştürülüyor (q4_k_m)...")
# Bu işlem sırasında oluşan 16GB'lık geçici dosya artık sorun olmayacak
model.save_pretrained_gguf("temp_model_gguf", tokenizer, quantization_method = "q4_k_m")

# 7. Final Dosyayı Output'a Taşı
# Sadece sonuç dosyasını (4-5 GB) Kaggle'ın görebileceği yere taşıyoruz
source_file = "temp_model_gguf/temp_model_gguf-unsloth.Q4_K_M.gguf"
dest_file = "/kaggle/working/finwise_scribe_v1.gguf"

print("📦 Dosya Output klasörüne taşınıyor...")
if os.path.exists(source_file):
    shutil.copy(source_file, dest_file)
    print(f"\n✅ İŞLEM BAŞARILI!")
    print(f"Dosya hazır: {dest_file}")
    print("Sağ taraftaki 'Output' panelinden dosyayı indirebilirsin.")
else:
    print("❌ Hata: GGUF dosyası oluşturulamadı.")

Çalışma dizini /tmp olarak değiştirildi.
Dosya indiriliyor (ID: 1qDta8PKGAWzanCn8_HXcZY2YRwwxWVJp)...
Downloading...
From (original): https://drive.google.com/uc?id=1qDta8PKGAWzanCn8_HXcZY2YRwwxWVJp
From (redirected): https://drive.google.com/uc?id=1qDta8PKGAWzanCn8_HXcZY2YRwwxWVJp&confirm=t&uuid=1c86150c-ebe8-4a41-882a-6fb630bc6b3f
To: /tmp/finwise_scribe_adapter_v1.zip
100%|████████████████████████████████████████| 158M/158M [00:01<00:00, 86.1MB/s]
Zip açılıyor...
Archive:  finwise_scribe_adapter_v1.zip
   creating: finwise_adapter/finwise_scribe_adapter/
  inflating: finwise_adapter/finwise_scribe_adapter/tokenizer_config.json  
  inflating: finwise_adapter/finwise_scribe_adapter/adapter_config.json  
  inflating: finwise_adapter/finwise_scribe_adapter/README.md  
  inflating: finwise_adapter/finwise_scribe_adapter/adapter_model.safetensors  
  inflating: finwise_adapter/finwise_scribe_adapter/special_tokens_map.json  
  inflating: finwise_adapter/finwise_scribe_adapter/tokenizer.js

/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['target_parameters'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.11.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


💾 GGUF formatına dönüştürülüyor (q4_k_m)...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [01:00<00:00, 15.03s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [02:10<00:00, 32.60s/it]


Unsloth: Merge process complete. Saved to `/tmp/temp_model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['llama-3-8b.F16.gguf']
Unsloth: [2] Conv

In [ ]:
import os
import shutil

print("🔍 Kayıp GGUF dosyası aranıyor...")

# /tmp klasörünü tara
found_file = None
for root, dirs, files in os.walk("/tmp"):
    for file in files:
        if file.endswith(".gguf"):
            found_file = os.path.join(root, file)
            print(f"✅ BULUNDU: {found_file}")
            break
    if found_file: break

# Dosyayı Output'a taşı
if found_file:
    dest_path = "/kaggle/working/finwise_scribe_v1.gguf"
    print(f"📦 Dosya taşınıyor: {dest_path} ...")
    shutil.copy(found_file, dest_path)
    print("\n🎉 İŞLEM TAMAM! Sağ taraftaki 'Output' panelini yenile (Refresh).")
    print("Dosyayı oradan indirebilirsin.")
else:
    print("❌ Hata: GGUF dosyası /tmp içinde bulunamadı.")
    # Debug için /tmp içeriğini listeleyelim
    print("DEBUG: /tmp içeriği:")
    print(os.listdir("/tmp"))

🔍 Kayıp GGUF dosyası aranıyor...
✅ BULUNDU: /tmp/llama-3-8b.Q4_K_M.gguf
📦 Dosya taşınıyor: /kaggle/working/finwise_scribe_v1.gguf ...

🎉 İŞLEM TAMAM! Sağ taraftaki 'Output' panelini yenile (Refresh).
Dosyayı oradan indirebilirsin.


In [ ]:
# --- KAGGLE to HUGGING FACE UPLOAD ---
from huggingface_hub import HfApi
import os

# 1. Token'ını Buraya Yapıştır (Read/Write izni olan token lazım)
# Eğer 'Write' izni yoksa, HF ayarlarından yeni bir 'Write' token al.
HF_TOKEN = "" # <-- BURAYA TOKEN'INI YAPIŞTIR

try:
    api = HfApi(token=HF_TOKEN)
    
    # Kullanıcı adını otomatik bul
    user_info = api.whoami()
    username = user_info["name"]
    print(f"👤 Giriş yapıldı: {username}")

    # 2. Yeni bir Model Reposu Oluştur
    repo_name = "finwise-scribe-model"
    repo_id = f"{username}/{repo_name}"
    
    print(f"🔨 Repo hazırlanıyor: {repo_id} ...")
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

    # 3. Dosyayı Yükle
    file_path = "/kaggle/working/finwise_scribe_v1.gguf"
    
    if not os.path.exists(file_path):
        print(f"❌ HATA: Dosya bulunamadı: {file_path}")
        # Belki dosya hala /tmp içindedir? Kontrol edelim
        if os.path.exists("/tmp/llama-3-8b.Q4_K_M.gguf"):
            print("⚠️ Dosya /tmp içinde bulundu, oradan yükleniyor...")
            file_path = "/tmp/llama-3-8b.Q4_K_M.gguf"
        else:
            raise FileNotFoundError("GGUF dosyası ne working ne de tmp klasöründe yok!")

    print(f"🚀 Yükleme başlıyor: {file_path} -> Hugging Face Hub")
    print("Bu işlem dosya boyutuna (4-5GB) göre 2-5 dakika sürebilir...")
    
    api.upload_file(
        path_or_fileobj=file_path,
        path_in_repo="finwise_scribe_v1.gguf",
        repo_id=repo_id,
        repo_type="model"
    )
    
    print("\n✅ BAŞARILI! Dosyan bulutta güvende.")
    print(f"📥 İndirme Linkin: https://huggingface.co/{repo_id}/blob/main/finwise_scribe_v1.gguf")

except Exception as e:
    print(f"\n❌ Bir hata oluştu: {e}")
    print("İpucu: Token'ının 'WRITE' (Yazma) izni olduğundan emin ol.")

👤 Giriş yapıldı: MV17
🔨 Repo hazırlanıyor: MV17/finwise-scribe-model ...
🚀 Yükleme başlıyor: /kaggle/working/finwise_scribe_v1.gguf -> Hugging Face Hub
Bu işlem dosya boyutuna (4-5GB) göre 2-5 dakika sürebilir...

✅ BAŞARILI! Dosyan bulutta güvende.
📥 İndirme Linkin: https://huggingface.co/MV17/finwise-scribe-model/blob/main/finwise_scribe_v1.gguf
